# LC 5 — Longest Palindromic Substring
**Day 55 | String DP / Palindromes | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> A palindrome expands symmetrically from
its center. For every character (and every gap between characters),
try to grow outward as long as both sides match. The widest expansion
wins.
</div>

## Official Problem Statement

Given a string `s`, return the longest palindromic substring in `s`.

**Constraints:**
- `1 <= s.length <= 1000`
- `s` consists of only digits and English letters.

## What This Is Actually Asking

Find the contiguous slice of `s` that reads the same forwards and
backwards, and is as long as possible.
A single character is always a palindrome of length 1.
There may be ties; returning any one correct answer is accepted.
The naive approach checks all O(n²) substrings and validates each
in O(n) — we need better.
Expand-around-center gives us O(n²) time with O(1) extra space.

## Walk Through an Example by Hand

```
s = "babad"

Center i=0 ('b'):
  odd : expand(0,0) -> 'b'           len=1
  even: expand(0,1) -> s[0]!=s[1]    stop

Center i=1 ('a'):
  odd : expand(1,1) -> 'a'           len=1
        expand(0,2) -> s[0]=='b'==s[2]? no -> stop (b!=b wait...)
        actually b==b -> 'bab'        len=3  <-- new best
  even: expand(1,2) -> s[1]!=s[2]    stop

Center i=2 ('b'):
  odd : expand(2,2) -> 'b'           len=1
        expand(1,3) -> s[1]='a'==s[3]='a' -> 'aba'  len=3  (tie)
  even: expand(2,3) -> s[2]!=s[3]    stop

Centers i=3,4: no improvement.
Result: 'bab' (or 'aba')  length=3
```

## The Picture

```
s =  b  a  b  a  d
idx  0  1  2  3  4

Odd center at i=1:

         center
           |
     <--  [a]  -->
      l=1     r=1

step 1:  l=0, r=2  ->  s[0]='b' == s[2]='b'  match!

     <--[b  a  b]-->
      l=0       r=2

step 2:  l=-1 -> out of bounds, STOP
         palindrome = s[0:3] = 'bab'

Even center between i=1 and i=2:

         | |
     <--[a  b]-->
      l=1     r=2
      s[1]='a' != s[2]='b'  -> STOP immediately
```

## When To Use This Pattern

- When the problem involves **palindrome detection** in a string,
  think **expand-around-center**.
- When you need to find the **longest** palindrome, think **track
  best span** across all 2n-1 centers.
- When constraints are n ≤ 1000 and O(n²) is acceptable, think
  **simple expand** over Manacher's algorithm.
- When a DP table would use O(n²) space, think **expand** for O(1)
  space instead.
- When multiple valid answers exist, think **return any**.

## The Approach

Iterate over every index as an odd center and every adjacent pair
as an even center — that covers all 2n-1 possible centers.
At each center, expand left and right simultaneously while characters
match and indices stay in bounds.
After each expansion, compare the resulting length to the current
best and update the recorded start index if longer.
Return the slice corresponding to the best span.

In [1]:
# No imports needed beyond builtins


In [2]:
def test_harness(func):
    """
    Validate longest palindromic substring.
    Checks: result is a palindrome, length matches expected,
    result is a substring of s.
    """
    cases = [
        ("babad",  3),
        ("cbbd",   2),
        ("a",      1),
        ("ac",     1),
        ("racecar", 7),
        ("abacaba", 7),
        ("aabbaa",  6),
    ]

    passed = 0
    for s, exp_len in cases:
        result = func(s)
        is_palindrome = result == result[::-1]
        correct_len  = len(result) == exp_len
        is_substring = result in s
        ok = is_palindrome and correct_len and is_substring
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"{status} s={s!r} "
                f"got={result!r} "
                f"palindrome={is_palindrome} "
                f"len_ok={correct_len}"
            )
    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")


In [7]:
def longest_palindrome(s: str) -> str:
    """
    Return the longest palindromic substring of s.

    Strategy: expand-around-center.
    For each of the 2n-1 centers, expand outward while
    characters match. Track the longest span found.

    Args:
        s: input string, 1 <= len(s) <= 1000

    Returns:
        One longest palindromic substring (any if tied).
    """
    res = ""
    def traverseIt(ss, l, r, result):
        while l>=0 and r <= len(ss) -1 and ss[r] == ss[l]:
            if r-l+1 > len(result):
                result = ss[l:r+1]
            l-=1
            r+=1
        return result
    for i in range(len(s)):
        res = traverseIt(s,i,i, res)
        res = traverseIt(s,i,i+1, res)
    return res
        
print(longest_palindrome("babad"))    # 'bab' or 'aba' (len 3)
print(longest_palindrome("cbbd"))     # 'bb'
print(longest_palindrome("racecar"))  # 'racecar'
print(longest_palindrome("a"))        # 'a'
print(longest_palindrome("aaaa"))     # 'aaaa'
test_harness(longest_palindrome)
    
        


bab
bb
racecar
a
aaaa

Result: 7/7 passed
All tests PASSED!


In [ ]:
# Uncomment and run when solution is ready
# test_harness(longest_palindrome)


## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (check all substrings) | O(n³) | O(1) | Validate each in O(n) |
| DP table | O(n²) | O(n²) | dp[i][j]=palindrome |
| **Expand-around-center** | **O(n²)** | **O(1)** | Optimal in practice |
| Manacher's algorithm | O(n) | O(n) | Linear but complex |


## Real World Connection

At Citi, detecting palindromic patterns in transaction ID sequences
can surface encoding anomalies or duplicate-reversal errors in
payment rails.
In AWS data pipelines, finding the longest repeated-mirror segment
in log keys helps identify misconfigured routing rules that create
symmetric error bursts.
For data engineers, palindrome-style expand logic underlies interval
merging: grow a window symmetrically until a boundary condition
breaks — the same left/right pointer discipline applies.
The O(1) space property of expand-around-center is attractive in
streaming contexts where allocating an O(n²) DP table is
prohibitive.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra